# Img Classification CNN 

### install first
pip3’ install numpy pillow torch torchvision

pip install torchvision

In [33]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [34]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [35]:
# Download and load CIFAR-10 dataset
# torchvision will automatically download if not present
data_dir = './data'
train_data = torchvision.datasets.CIFAR10(root=data_dir, train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root=data_dir, train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

Training samples: 50000
Test samples: 10000


In [36]:
# Debug: Check what the kernel can see
import os

test_path = '/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data/cifar-10-python.tar.gz'
print(f"File exists check: {os.path.exists(test_path)}")
print(f"Current working directory: {os.getcwd()}")
print(f"Listing data dir:")
if os.path.exists('/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data'):
    print(os.listdir('/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data'))
else:
    print("Data directory not found!")

File exists check: True
Current working directory: /home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN
Listing data dir:
['cifar-10-batches-py', 'OmniShotCut_ckpt.pth', 'cifar-10-python.tar.gz']


In [37]:
image, label =  train_data[0]

In [38]:
image.size()

torch.Size([3, 32, 32])

In [39]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [40]:
class NeuralNetwork(nn.Module):

    def __init__(self):
        super(NeuralNetwork, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 12, 5) # (12, 28, 28)
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14) -> (24, 5, 5) after conv2+pool
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 10, 10) -> (24, 5, 5) -> Flatten to (24*5*5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)  # Fixed: 24*5*5 = 600 (not 24*10*10)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [41]:
net = NeuralNetwork()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [42]:
# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)
print(f'Training on {device}')

for epoch in range(30):
    print(f'Training Epoch {epoch}...')

    running_loss = 0.0
    
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch} Loss: {running_loss / len(train_loader):.4f}')

print('Training finished!')

Training on cpu
Training Epoch 0...


Epoch 0 Loss: 2.2642
Training Epoch 1...
Epoch 1 Loss: 1.8775
Training Epoch 2...
Epoch 2 Loss: 1.5829
Training Epoch 3...
Epoch 3 Loss: 1.4320
Training Epoch 4...
Epoch 4 Loss: 1.3282
Training Epoch 5...
Epoch 5 Loss: 1.2426
Training Epoch 6...
Epoch 6 Loss: 1.1717
Training Epoch 7...
Epoch 7 Loss: 1.1109
Training Epoch 8...
Epoch 8 Loss: 1.0576
Training Epoch 9...
Epoch 9 Loss: 1.0092
Training Epoch 10...
Epoch 10 Loss: 0.9689
Training Epoch 11...
Epoch 11 Loss: 0.9330
Training Epoch 12...
Epoch 12 Loss: 0.8965
Training Epoch 13...
Epoch 13 Loss: 0.8631
Training Epoch 14...
Epoch 14 Loss: 0.8298
Training Epoch 15...
Epoch 15 Loss: 0.8035
Training Epoch 16...
Epoch 16 Loss: 0.7735
Training Epoch 17...
Epoch 17 Loss: 0.7448
Training Epoch 18...
Epoch 18 Loss: 0.7210
Training Epoch 19...
Epoch 19 Loss: 0.6984
Training Epoch 20...
Epoch 20 Loss: 0.6713
Training Epoch 21...
Epoch 21 Loss: 0.6497
Training Epoch 22...
Epoch 22 Loss: 0.6236
Training Epoch 23...
Epoch 23 Loss: 0.6057
Training

In [43]:
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize to 32x32 pixels (use tuple)
    transforms.ToTensor(), # Convert to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize to [-1, 1]
])

def load_image(image_path):
    image = Image.open(image_path)

In [44]:
net = NeuralNetwork()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [45]:
correct = 0
total = 0

net.eval()
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f}%')

Accuracy of the network on the 10000 test images: 68.21%


In [46]:
# Note: Model accuracy is ~68%, so some predictions will be wrong
# To improve accuracy: more epochs, deeper network, dropout, data augmentation, or transfer learning

new_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize to 32x32 pixels
    transforms.ToTensor(), # Convert to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize to [-1, 1]
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image) # Apply the transformation
    image = image.unsqueeze(0) # Add batch dimension
    return image

image_path = ['example1.jpg', 'example2.jpg']  # Replace with your image paths
images = [load_image(img) for img in image_path]

net.eval()
with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction: {class_names[predicted.item()]}')

Prediction: airplane
Prediction: horse
